### Load the Saved Datasets :

In [2]:
from pathlib import Path

print("Current Working Directory:")
print(Path.cwd())

print("\nFiles in Current Directory:")
for file in Path.cwd().iterdir():
    print(file.name)

print("\nSearching for Saved CSV Files:")

project_path = Path(
    r"C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics"
)

for filename in [
    "model_predictors.csv",
    "model_target.csv",
    "feature_engineered_dataset.csv"
]:
    matches = list(project_path.rglob(filename))

    print(f"\n{filename}:")
    for match in matches:
        print(match)

Current Working Directory:
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\02_machine_learning

Files in Current Directory:
.ipynb_checkpoints
01_ml_problem_definition.ipynb
02_user_level_dataset_preparation.ipynb
03_ml_data_preprocessing.ipynb
04_ml_eda.ipynb
05_feature_engineering.ipynb
06_train_test_split.ipynb

Searching for Saved CSV Files:

model_predictors.csv:
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\model_predictors.csv

model_target.csv:
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\model_target.csv

feature_engineered_dataset.csv:
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\feature_engineered_dataset.csv


### Load Using the Actual File Location :

In [3]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics"
)

def find_file(filename):
    matches = list(PROJECT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found inside {PROJECT_DIR}"
        )

    return matches[0]

predictors_path = find_file("model_predictors.csv")
target_path = find_file("model_target.csv")
dataset_path = find_file("feature_engineered_dataset.csv")

X = pd.read_csv(predictors_path)
y = pd.read_csv(target_path).squeeze("columns")
full_dataset = pd.read_csv(dataset_path)

full_dataset["index_date"] = pd.to_datetime(
    full_dataset["index_date"]
)

print("Predictor Shape:", X.shape)
print("Target Shape:", y.shape)
print("Full Dataset Shape:", full_dataset.shape)

print("\nFiles Loaded From:")
print(predictors_path)
print(target_path)
print(dataset_path)

Predictor Shape: (127219, 50)
Target Shape: (127219,)
Full Dataset Shape: (127219, 68)

Files Loaded From:
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\model_predictors.csv
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\model_target.csv
C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\python\notebooks\data\processed\feature_engineered_dataset.csv


### Inspect Cohort Dates for Chronological Splitting :

In [4]:
# Step 1: Inspect Cohort Dates

print("Earliest Index Date:")
print(full_dataset["index_date"].min())

print("\nLatest Index Date:")
print(full_dataset["index_date"].max())

print("\nUnique Cohort Dates:")
print(full_dataset["index_date"].nunique())

# Records per cohort date
cohort_distribution = (
    full_dataset
    .groupby("index_date")
    .agg(
        total_records=("user_id", "count"),
        positive_cases=("feature_adopted", "sum")
    )
    .reset_index()
)

cohort_distribution["negative_cases"] = (
    cohort_distribution["total_records"]
    - cohort_distribution["positive_cases"]
)

cohort_distribution["adoption_rate"] = (
    cohort_distribution["positive_cases"]
    / cohort_distribution["total_records"]
    * 100
).round(2)

print("\nCohort Distribution:")
display(cohort_distribution)

print("\nTotal Cohort Records:")
print(cohort_distribution["total_records"].sum())

print("\nTotal Positive Cases:")
print(cohort_distribution["positive_cases"].sum())

Earliest Index Date:
2024-03-01 00:00:00

Latest Index Date:
2026-09-17 00:00:00

Unique Cohort Dates:
32

Cohort Distribution:


,index_date,total_records,positive_cases,negative_cases,adoption_rate
0,2024-03-01,29,1,28,3.45
1,2024-03-31,118,5,113,4.24
2,2024-04-30,250,18,232,7.20
3,2024-05-30,438,28,410,6.39
4,2024-06-29,687,44,643,6.40
5,2024-07-29,996,54,942,5.42
6,2024-08-28,1299,64,1235,4.93
7,2024-09-27,1584,73,1511,4.61
8,2024-10-27,1909,69,1840,3.61
9,2024-11-26,2180,89,2091,4.08



Total Cohort Records:
127219

Total Positive Cases:
1993


### Create the Chronological Train-Test Split :

In [5]:
# Step 2: Create Chronological Train-Test Split

# Sort unique cohort dates
cohort_dates = sorted(
    full_dataset["index_date"].unique()
)

# Use the first 24 dates for training
split_position = 24

train_dates = cohort_dates[:split_position]
test_dates = cohort_dates[split_position:]

# Create masks
train_mask = full_dataset["index_date"].isin(train_dates)
test_mask = full_dataset["index_date"].isin(test_dates)

# Split predictors and target using the same row positions
X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training Period:")
print(min(train_dates), "to", max(train_dates))

print("\nTesting Period:")
print(min(test_dates), "to", max(test_dates))

print("\nTraining Shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting Shapes:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining Target Distribution:")
display(y_train.value_counts())

print("\nTesting Target Distribution:")
display(y_test.value_counts())

Training Period:
2024-03-01 00:00:00 to 2026-01-20 00:00:00

Testing Period:
2026-02-19 00:00:00 to 2026-09-17 00:00:00

Training Shapes:
X_train: (70577, 50)
y_train: (70577,)

Testing Shapes:
X_test: (56642, 50)
y_test: (56642,)

Training Target Distribution:


feature_adopted
0    68957
1     1620
Name: count, dtype: int64


Testing Target Distribution:


feature_adopted
0    56269
1      373
Name: count, dtype: int64

### Check User Overlap Between Training and Testing :

In [6]:
# Step 3: Check User Overlap

train_users = set(
    full_dataset.loc[train_mask, "user_id"]
)

test_users = set(
    full_dataset.loc[test_mask, "user_id"]
)

overlapping_users = train_users.intersection(test_users)

print("Unique Training Users:", len(train_users))
print("Unique Testing Users:", len(test_users))
print("Overlapping Users:", len(overlapping_users))

print(
    "User Overlap Percentage:",
    round(
        len(overlapping_users) / len(test_users) * 100,
        2
    )
)

Unique Training Users: 7921
Unique Testing Users: 7572
Overlapping Users: 6301
User Overlap Percentage: 83.21


### Validate the chronological split :

In [7]:
# Step 4: Final Train-Test Split Validation

# Check date separation
train_max_date = full_dataset.loc[
    train_mask, "index_date"
].max()

test_min_date = full_dataset.loc[
    test_mask, "index_date"
].min()

print("Last Training Date:", train_max_date)
print("First Testing Date:", test_min_date)

print(
    "Chronological Separation:",
    train_max_date < test_min_date
)

# Check row counts
print("\nRow Count Validation:")
print("Training rows:", len(X_train))
print("Training target rows:", len(y_train))
print("Testing rows:", len(X_test))
print("Testing target rows:", len(y_test))

# Check target classes
print("\nTraining Classes:")
print(sorted(y_train.unique()))

print("\nTesting Classes:")
print(sorted(y_test.unique()))

# Check missing values
print("\nMissing Values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

Last Training Date: 2026-01-20 00:00:00
First Testing Date: 2026-02-19 00:00:00
Chronological Separation: True

Row Count Validation:
Training rows: 70577
Training target rows: 70577
Testing rows: 56642
Testing target rows: 56642

Training Classes:
[np.int64(0), np.int64(1)]

Testing Classes:
[np.int64(0), np.int64(1)]

Missing Values:
X_train: 14967
X_test: 1122


### Identify missing values by column :

In [8]:
# Step 5: Analyze Missing Values by Column

train_missing = X_train.isna().sum()
test_missing = X_test.isna().sum()

missing_summary = pd.DataFrame({
    "train_missing": train_missing,
    "train_missing_pct": (
        train_missing / len(X_train) * 100
    ),
    "test_missing": test_missing,
    "test_missing_pct": (
        test_missing / len(X_test) * 100
    )
})

missing_summary = missing_summary[
    (missing_summary["train_missing"] > 0) |
    (missing_summary["test_missing"] > 0)
].sort_values(
    by="train_missing",
    ascending=False
)

display(missing_summary.round(2))

,train_missing,train_missing_pct,test_missing,test_missing_pct
subscription_plan_type,4989,7.07,374,0.66
subscription_monthly_price,4989,7.07,374,0.66
subscription_tenure_days,4989,7.07,374,0.66


### Verify the reason for missing subscription values :

In [9]:
# Step 6: Validate Subscription Missing-Value Logic

subscription_columns = [
    "subscription_plan_type",
    "subscription_monthly_price",
    "subscription_tenure_days"
]

train_rows = full_dataset.loc[train_mask].copy()
test_rows = full_dataset.loc[test_mask].copy()

for name, dataset, mask in [
    ("Training", train_rows, train_mask),
    ("Testing", test_rows, test_mask)
]:
    print(f"\n{name} Dataset")

    not_started = dataset["subscription_started"].eq(0)

    print(
        "Users without a started subscription:",
        not_started.sum()
    )

    for column in subscription_columns:
        missing = dataset[column].isna()

        print(f"\n{column}")
        print("Missing values:", missing.sum())
        print(
            "Missing values where subscription not started:",
            (missing & not_started).sum()
        )
        print(
            "Missing values where subscription started:",
            (missing & ~not_started).sum()
        )


Training Dataset
Users without a started subscription: 4989

subscription_plan_type
Missing values: 4989
Missing values where subscription not started: 4989
Missing values where subscription started: 0

subscription_monthly_price
Missing values: 4989
Missing values where subscription not started: 4989
Missing values where subscription started: 0

subscription_tenure_days
Missing values: 4989
Missing values where subscription not started: 4989
Missing values where subscription started: 0

Testing Dataset
Users without a started subscription: 374

subscription_plan_type
Missing values: 374
Missing values where subscription not started: 374
Missing values where subscription started: 0

subscription_monthly_price
Missing values: 374
Missing values where subscription not started: 374
Missing values where subscription started: 0

subscription_tenure_days
Missing values: 374
Missing values where subscription not started: 374
Missing values where subscription started: 0


### Save Train-Test Datasets :

In [10]:
# Step 8: Save Train-Test Datasets

from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save predictors
X_train.to_csv(
    OUTPUT_DIR / "X_train.csv",
    index=False
)

X_test.to_csv(
    OUTPUT_DIR / "X_test.csv",
    index=False
)

# Save target
y_train.to_csv(
    OUTPUT_DIR / "y_train.csv",
    index=False
)

y_test.to_csv(
    OUTPUT_DIR / "y_test.csv",
    index=False
)

# Save cohort metadata for reproducibility
full_dataset.loc[
    train_mask,
    ["user_id", "index_date", "feature_adopted"]
].to_csv(
    OUTPUT_DIR / "train_cohort_metadata.csv",
    index=False
)

full_dataset.loc[
    test_mask,
    ["user_id", "index_date", "feature_adopted"]
].to_csv(
    OUTPUT_DIR / "test_cohort_metadata.csv",
    index=False
)

print("All train-test files saved successfully.")

for file in OUTPUT_DIR.glob("*.csv"):
    print(file.name, ":", file.exists())

All train-test files saved successfully.
api_monitoring_adoption_summary.csv : True
api_usage_features.csv : True
categorical_value_report.csv : True
data_type_report.csv : True
date_type_report.csv : True
duplicate_summary.csv : True
feature_engineered_dataset.csv : True
feature_engineering_date_validation.csv : True
historical_user_cohort_dataset.csv : True
missing_value_report.csv : True
missing_value_summary.csv : True
ml_eda_api_adoption_comparison.csv : True
ml_eda_dataset_summary.csv : True
ml_eda_engagement_adoption_comparison.csv : True
ml_eda_feature_distribution.csv : True
ml_eda_subscription_distribution.csv : True
ml_eda_user_api_activity.csv : True
ml_eda_user_engagement.csv : True
model_predictors.csv : True
model_target.csv : True
numerical_outlier_report.csv : True
preprocessing_summary.csv : True
primary_key_duplicate_report.csv : True
product_event_features.csv : True
test_cohort_metadata.csv : True
train_cohort_metadata.csv : True
X_test.csv : True
X_train.csv : Tru